# 소비기한 추출 — [DAT] 캐치캐치

상품 뒷면 이미지에서 소비기한을 추출해 `submission.csv` 를 생성한다.

**파이프라인 요약:**
1. **이미지 로딩 & 정규화:** EXIF 방향 보정 및 효율적 디코딩
2. **검출:** 날짜 전용 검출기(NanoDet-Plus-m) 또는 범용 텍스트 검출기
3. **박스 필터링:** 기하학적 제약으로 노이즈 후보 필터링
4. **인식:** PP-OCRv6 small 인식기로 텍스트 크롭 배치 추론
5. **파싱 & 선별:** 다중 정규식 매칭 및 규칙 기반 최적 날짜 선별
6. **출력:** 스키마 검증 및 submission.csv 저장


In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

## 1. 환경 설정 및 스레드 고정

일관된 추론 속도 및 재현성을 위해 4-Core vCPU 환경에 맞춰 스레드 수를 사전에 고정한다.


In [ ]:
import sys, time, csv, re
from pathlib import Path

T0 = time.time()

# --- 스레드 고정: 반드시 cv2 / onnxruntime import 보다 먼저 ---
THREADS = 4                                   # 채점 환경 = Standard 4-Core vCPU
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(THREADS)

# --- 루트 리졸버 ---
# 채점기가 저장소 밖에서 nbconvert 를 부르면 cwd 가 저장소 루트가 아닐 수 있다.
# itda_ocr/ 를 담은 디렉터리를 위로 훑어 찾는다.
def _find_root():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "itda_ocr" / "pipeline.py").exists():
            return base
    raise RuntimeError("itda_ocr 패키지를 찾지 못했습니다. 저장소 루트에서 실행해 주세요.")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from itda_ocr.pipeline import Config, iter_images, run

# 2400초 제한. 여유를 두고 마감을 잡아 저장·검증 시간을 남긴다.
DEADLINE = T0 + 2100
print(f"root={ROOT}  threads={THREADS}  budget={DEADLINE - T0:.0f}s")

## 2. 설정

임계값을 코드에 흩뿌리지 않고 한 곳에 모았다. 각 값의 근거는 주석에 있다.

In [ ]:
CFG = Config(
    draft_to=720,             # 이미지 로드 디코딩 크기
    det_side=640,             # 검출 입력 크기
    top_k=2,                  # 인식 배치 크기
    max_k=9,                  # 최대 크롭 수 상한
    threads=THREADS,          # 스레드 수
    impute_missing=False,     # 결측값 임의 보정 비활성화
    box_thresh=0.5,
    unclip_ratio=1.6,
    per_image_budget=0.15,
    flush_every=50,
    nanodet_onnx=(lambda p: str(p) if p.exists() else None)(ROOT / "weights" / "date_detector_ema.onnx"),
    nanodet_score_thr=0.05,
    nanodet_expand=0.10,      # 크롭 여백 확장 비율
    crop_tta=0.20,            # 박스마다 (원래 여백, +20%) 두 크롭을 읽어 절단 판독을 회수
    det_fallback=True,        # 검출 0개인 이미지만 범용 DB 검출로 재시도 (발동률 0.5%)
    prefetch=True,            # 다음 이미지 디코딩을 보조 스레드로 선행 (-5.9%, 예측 불변)
    rec_onnx=(lambda p: str(p) if p.exists() else None)(ROOT / "weights" / "ppocrv6_rec_small.onnx"),
    rec_fallback=False,       # crop_tta 도입으로 중복(+0.05)이라 끔 — 꼬리 지연만 커진다
)

from itda_ocr.engine import Engine
_t = time.time()
ENGINE = Engine(det_side=CFG.det_side, threads=CFG.threads,
                box_thresh=CFG.box_thresh, unclip_ratio=CFG.unclip_ratio,
                nanodet_onnx=CFG.nanodet_onnx,
                nanodet_score_thr=CFG.nanodet_score_thr,
                nanodet_expand=CFG.nanodet_expand,
                rec_onnx=CFG.rec_onnx, rec_fallback=CFG.rec_fallback)
_det_name = "NanoDet 날짜검출기" if CFG.nanodet_onnx else "RapidOCR 범용검출"
_rec_name = "PP-OCRv6 small" if CFG.rec_onnx else "PP-OCRv4 mobile (번들)"
if CFG.crop_tta:
    _rec_name += f" + 크롭 TTA(+{CFG.crop_tta:.0%})"
if ENGINE.has_fallback:
    _rec_name += " + PP-OCRv4 폴백"
print(f"engine ready in {time.time() - _t:.2f}s  |  검출: {_det_name}  |  인식: {_rec_name}  "
      f"(오프라인 로컬 가중치 준비 완료)")


## 3~4. 순차 실행 및 결과 저장

`run()`은 시작 즉시 빈 유효 CSV를 생성하여 장애 발생 시에도 결과를 보장하며,
각 이미지를 순차적으로 처리하여 최종 결과를 파일에 기록한다.


In [ ]:
N = len(iter_images(INPUT_DIR))
print(f"{N}장  장당 예산 {(DEADLINE - time.time()) / max(N, 1) * 1000:.0f} ms")

RESULT = run(INPUT_DIR, OUTPUT_PATH, CFG, engine=ENGINE, deadline=DEADLINE)

## 6. 스키마 검증

**이 셀은 어떤 경우에도 raise하지 않는다.** 형식이 어긋나면 안전값으로 되돌리고
경고만 남긴다 — 예외를 던지면 정량 60점이 통째로 0점이 된다.

In [ ]:
PAT = {"year": re.compile(r"^\d{4}$"), "month": re.compile(r"^\d{2}$"),
       "day": re.compile(r"^\d{2}$"),
       "final_date": re.compile(r"^\d{4}-\d{2}-\d{2}$")}
FIELDS = ["image_id", "year", "month", "day", "final_date"]

rows, repaired = RESULT["rows"], 0
for r in rows:
    for key, pat in PAT.items():
        v = r.get(key, "NONE")
        if not isinstance(v, str):
            v = str(v)
        if v != "NONE" and not pat.match(v):
            v, repaired = "NONE", repaired + 1
        r[key] = v
    # 불변식: final_date 와 y/m/d 는 서로 모순되면 안 된다.
    ymd = (r["year"], r["month"], r["day"])
    if r["final_date"] != "NONE" and r["final_date"] != "-".join(ymd):
        r["final_date"], repaired = "NONE", repaired + 1

with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

if len(rows) != N:
    print(f"경고: 행 수 불일치 ({len(rows)} != {N})")
print(f"saved {len(rows)} rows to {OUTPUT_PATH}" + (f"  (복구 {repaired}건)" if repaired else ""))

## 7. 실행 리포트

전체 실행 시간 및 세부 처리 통계를 출력한다.


In [ ]:
el = RESULT["elapsed"]
print(f"이미지        {RESULT['n']}장")
print(f"경과          {el:.1f}s  (모델 로드 제외)")
print(f"장당          {RESULT['ms_per_image']:.0f} ms")
print(f"커버리지      {RESULT['coverage']:.1%}   (날짜를 낸 비율)")
print(f"단계 하향     {RESULT['degraded']}장")
print(f"예산 초과 생략 {RESULT['skipped']}장")
print(f"전체 벽시계   {time.time() - T0:.1f}s / 2400s")